
# Chapter 12
## Neural Networks That Write Like Shakespeare: Recurrent Layers for Variable-Length Data

This chapter explores how recurrent neural networks process sequences of words with variable lengths.

The chapter introduces word embeddings, sentence representations, recurrent transition matrices, forward propagation, backpropagation, and training a neural network to predict the next word in a sequence.

The model is implemented using Python and NumPy.


In [ ]:

import sys
import random
import math
from collections import Counter
import numpy as np

np.random.seed(1)
random.seed(1)

print("NumPy version:", np.__version__)
print("Libraries imported successfully.")



## The Challenge of Arbitrary Length

A neural network needs a numerical representation of its input. In natural language processing, an input can consist of a single word, a phrase, or an entire sentence.

The challenge is that sentences do not all have the same length. Concatenating word embeddings produces vectors whose dimensions depend on the number of words in a sentence.

A recurrent neural network addresses this problem by processing words one at a time. It maintains a hidden representation that is updated whenever a new word is encountered.

This allows the network to process sequences of different lengths while reusing the same recurrent weights at each step.



## Do Comparisons Really Matter?

When comparing representations of sentences, it is useful to consider whether the representation preserves the information needed for the task.

Two sentences can contain similar words but express different meanings because the words appear in different orders.

A representation that ignores word order may capture some similarities, but it can lose information about the relationships between words.

This motivates the use of recurrent representations, which update their internal state as words are processed sequentially.



## The Surprising Power of Averaged Word Vectors

One simple way to represent a sentence is to average the embeddings of its words.

This produces a fixed-size vector regardless of the sentence length. It is computationally simple and can capture information about the words appearing in a sentence.

However, averaging does not preserve word order. Sentences containing the same words may receive similar representations even when their meanings differ.

The chapter uses this limitation to motivate a representation that changes as each word is processed.



## How Is Information Stored in These Embeddings?

A word embedding represents a word as a numerical vector. Each dimension can contribute to patterns learned from the training objective.

The meaning of an embedding is not manually assigned to each dimension. Instead, its values are adjusted during training so that the network becomes better at its prediction task.

For sentence processing, the network needs a representation that can incorporate information from multiple words rather than only one word at a time.



## How Does a Neural Network Use Embeddings?

A neural network can use word embeddings as numerical inputs. An embedding lookup selects the vector associated with a particular word.

In a recurrent model, the current word embedding is combined with the previous hidden representation. The resulting vector becomes the new hidden representation.

The hidden representation therefore depends on both the current word and information carried forward from earlier words.



## The Limitations of Bag-of-Words Vectors

A bag-of-words representation describes a sentence using the words it contains, without necessarily preserving their order.

This approach can be useful for identifying word occurrence, but it has an important limitation: the sequence in which words appear is not explicitly represented.

A recurrent network addresses this limitation by processing the sequence one word at a time. The hidden state changes at each step, allowing the order of processing to influence the final representation.



## Using Identity Vectors to Sum Word Embeddings

An identity matrix contains ones on its main diagonal and zeros elsewhere. Multiplying a one-hot vector by an embedding matrix selects the embedding associated with a word.

This provides a mathematical way to describe an embedding lookup. Each word is represented by an index, and that index selects the corresponding row of the embedding matrix.

In the recurrent model, the selected word embedding is added to the transformed previous hidden state.



## Matrices That Change Absolutely Nothing

An identity matrix leaves a vector unchanged when used in a compatible matrix multiplication.

The recurrent matrix is initially set to an identity matrix. At initialization, this allows the previous hidden representation to pass through the recurrent transformation unchanged.

During training, the recurrent matrix is updated. It can then learn a transformation that helps the network preserve and modify information from earlier words.



## Learning the Transition Matrices

The recurrent matrix controls how information from the previous hidden state contributes to the next hidden state.

The decoder matrix maps a hidden representation to scores for possible next words. Softmax converts these scores into probabilities.

Both matrices are trainable parameters. During learning, their values are adjusted based on the prediction error so that the network can improve its next-word predictions.



## Learning to Create Useful Sentence Vectors

The model begins with a trainable start vector representing an empty sequence. Each word updates the hidden representation through the recurrent transformation and its word embedding.

The prediction objective encourages the hidden representation to contain information useful for predicting the next word.

As training proceeds, the word embeddings, recurrent matrix, decoder matrix, and start vector are adjusted together. The resulting sentence representation is learned from the prediction task rather than manually specified.


In [2]:

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)


def predict(sent):
    layers = list()

    layer = {}
    layer["hidden"] = start
    layers.append(layer)

    loss = 0
    preds = list()

    for target_i in range(len(sent)):
        layer = {}

        layer["pred"] = softmax(
            layers[-1]["hidden"].dot(decoder)
        )

        loss += -np.log(
            layer["pred"][sent[target_i]] + 1e-12
        )

        layer["hidden"] = (
            layers[-1]["hidden"].dot(recurrent)
            + embed[sent[target_i]]
        )

        layers.append(layer)

    return layers, loss



## How Do You Backpropagate into This?

Backpropagation calculates how the prediction loss should influence the model's parameters.

For each time step, the output error is calculated by comparing the predicted word probabilities with the one-hot representation of the target word.

The output error contributes to the hidden-state error through the decoder matrix. Because the hidden state depends on earlier hidden states, the error must also be propagated backward through the recurrent connections.

This process allows the model to update parameters that affect predictions at multiple positions in the sequence.


In [11]:

import numpy as np

# Membuat vocabulary dari dataset
vocab = set()

for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)

word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

index2word = {i: word for word, i in word2index.items()}


def words2indices(sentence):
    return [word2index[word] for word in sentence]


def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)


# Inisialisasi parameter model
np.random.seed(1)

embed_size = 10

embed = (
    np.random.rand(len(vocab), embed_size) - 0.5
) * 0.1

recurrent = np.eye(embed_size)

start = np.zeros(embed_size)

decoder = (
    np.random.rand(embed_size, len(vocab)) - 0.5
) * 0.1

one_hot = np.eye(len(vocab))

print("Vocabulary:", len(vocab))
print("Start:", start.shape)
print("Embedding:", embed.shape)
print("Recurrent:", recurrent.shape)
print("Decoder:", decoder.shape)


Vocabulary: 82
Start: (10,)
Embedding: (82, 10)
Recurrent: (10, 10)
Decoder: (10, 82)


In [12]:

# Pilih satu baris yang memiliki token
sample = next(sent for sent in tokens if len(sent) > 1)

sent = words2indices(sample[1:])

layers, loss = predict(sent)

print("Contoh kalimat:", sample)
print("Panjang sequence:", len(sent))
print("Jumlah layer state:", len(layers))
print("Loss:", float(loss))
print("Hidden state terakhir:", layers[-1]["hidden"])


Contoh kalimat: ['mary', 'moved', 'to', 'the', 'bathroom.']
Panjang sequence: 4
Jumlah layer state: 5
Loss: 17.626072523374415
Hidden state terakhir: [-0.03833094  0.11357642  0.09394154 -0.05950487 -0.01613077 -0.05975241
 -0.01189114  0.01577255 -0.00491284 -0.08239541]


In [13]:

def backpropagate(sent, layers):
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]

        if layer_idx > 0:
            target = sent[layer_idx - 1]

            layer["output_delta"] = (
                layer["pred"] - one_hot[target]
            )

            new_hidden_delta = (
                layer["output_delta"].dot(decoder.T)
            )

            if layer_idx == len(layers) - 1:
                layer["hidden_delta"] = new_hidden_delta
            else:
                layer["hidden_delta"] = (
                    new_hidden_delta
                    + layers[layer_idx + 1][
                        "hidden_delta"
                    ].dot(recurrent.T)
                )
        else:
            layer["hidden_delta"] = (
                layers[layer_idx + 1][
                    "hidden_delta"
                ].dot(recurrent.T)
            )

    return layers


In [14]:

layers = backpropagate(sent, layers)

print("Backpropagation selesai.")
print(
    "Ukuran hidden delta:",
    layers[0]["hidden_delta"].shape
)
print(
    "Ukuran output delta:",
    layers[1]["output_delta"].shape
)


Backpropagation selesai.
Ukuran hidden delta: (10,)
Ukuran output delta: (82,)


In [15]:

def train_step(sent, alpha=0.001):
    global start, decoder, embed, recurrent

    layers, loss = predict(sent)
    layers = backpropagate(sent, layers)

    seq_len = float(len(sent))

    # Update start vector
    start -= (
        layers[0]["hidden_delta"] * alpha / seq_len
    )

    # Update decoder, embeddings, and recurrent matrix
    for layer_idx, layer in enumerate(layers[1:]):
        decoder -= (
            np.outer(
                layers[layer_idx]["hidden"],
                layer["output_delta"]
            ) * alpha / seq_len
        )

        embed_idx = sent[layer_idx]

        embed[embed_idx] -= (
            layer["hidden_delta"] * alpha / seq_len
        )

        recurrent -= (
            np.outer(
                layers[layer_idx]["hidden"],
                layer["hidden_delta"]
            ) * alpha / seq_len
        )

    return loss


In [16]:

iterations = 30000
alpha = 0.001

for iteration in range(iterations):
    sample_tokens = tokens[iteration % len(tokens)]

    # Gunakan kata setelah token pertama sebagai sequence
    sent = words2indices(sample_tokens[1:])

    if len(sent) == 0:
        continue

    loss = train_step(sent, alpha=alpha)

    if iteration % 1000 == 0:
        perplexity = np.exp(
            min(loss / len(sent), 50)
        )
        print(
            f"Iteration: {iteration}, "
            f"Perplexity: {perplexity:.4f}"
        )


Iteration: 0, Perplexity: 81.9835
Iteration: 1000, Perplexity: 81.7560
Iteration: 2000, Perplexity: 81.3421
Iteration: 3000, Perplexity: 80.5084
Iteration: 4000, Perplexity: 78.6764
Iteration: 5000, Perplexity: 73.8944
Iteration: 6000, Perplexity: 54.5281
Iteration: 7000, Perplexity: 30.9304
Iteration: 8000, Perplexity: 20.8498
Iteration: 9000, Perplexity: 19.4278
Iteration: 10000, Perplexity: 18.4948
Iteration: 11000, Perplexity: 17.3075
Iteration: 12000, Perplexity: 15.4238
Iteration: 13000, Perplexity: 12.4851
Iteration: 14000, Perplexity: 9.2213
Iteration: 15000, Perplexity: 7.3037
Iteration: 16000, Perplexity: 6.2539
Iteration: 17000, Perplexity: 5.6396
Iteration: 18000, Perplexity: 5.2442
Iteration: 19000, Perplexity: 4.9817
Iteration: 20000, Perplexity: 4.8048
Iteration: 21000, Perplexity: 4.6806
Iteration: 22000, Perplexity: 4.5847
Iteration: 23000, Perplexity: 4.5017
Iteration: 24000, Perplexity: 4.4227
Iteration: 25000, Perplexity: 4.3432
Iteration: 26000, Perplexity: 4.2620



## Execution and Output Analysis

During training, the model predicts the next word in each sequence. The loss measures the prediction error, while perplexity provides an alternative measure of how uncertain the model is about the target words.

A lower perplexity generally indicates that the model assigns higher probabilities to the observed words. However, the training result depends on the dataset, vocabulary, initialization, and optimization process.

The model learns four main parameter groups:

- **Word embeddings (`embed`)** represent individual words.
- **Recurrent matrix (`recurrent`)** transforms the previous hidden state.
- **Decoder matrix (`decoder`)** produces scores for predicting the next word.
- **Start vector (`start`)** represents the initial state before any word is processed.

## Summary

In this chapter, we explored how recurrent neural networks represent sequences of variable lengths.

We discussed the limitations of concatenation and bag-of-words representations, the role of word embeddings, and the use of a recurrent matrix to update the hidden state as words are processed.

We then implemented forward propagation, backpropagation, and weight updates in Python using NumPy.

The main lesson is that a recurrent neural network can reuse the same parameters across sequence positions while maintaining a hidden representation that carries information from earlier words. Training adjusts these parameters to improve next-word prediction.
